## 学习目标

本教程介绍两个新知识点：

- 如何使用 Triton 实现反向传播。
- 如何在多个 Program 之间完成并行归约。

## 简介

层标准化（LayerNorm）算子最先在 BA2016 中提出，旨在提高序列模型（例如 Transformers）或小 batch size 神经网络的性能。

它以向量 $x$ 作为输入，并生成与输入 shape 相同的向量 $y$ 作为输出。标准化是通过减去均值并除以 $x$ 的标准差来实现的。

标准化后，会应用带有权重 $w$ 和偏置 $b$ 的可学习线性变换。前向传播可以表示为：

$$
y = \frac{x - \mathbb{E}[x]}{\sqrt{\operatorname{Var}(x) + \epsilon}} \cdot w + b
$$

其中 ：ϵ 是加到分母上的一个小常数，以保证数值稳定性。 首先让我们看看前向传播的实现。

In [1]:
import torch
import triton
import triton.language as tl


try:
    # This is https://github.com/NVIDIA/apex, NOT the apex on PyPi, so it
    # should not be added to extras_require in setup.py.
    # 这是 https://github.com/NVIDIA/apex，不是 PyPi 的 apex，
    # 所以不应该加进 setup.py 的额外依赖中
    import apex
    HAS_APEX = True
except ModuleNotFoundError:
    HAS_APEX = False




@triton.jit
def _layer_norm_fwd_fused(
    X,  # pointer to the input 输入指针
    Y,  # pointer to the output 输出指针
    W,  # pointer to the weights 权重指针
    B,  # pointer to the biases 偏差指针
    Mean,  # pointer to the mean 均值指针
    Rstd,  # pointer to the 1/std 1/std 指针
    stride,  # how much to increase the pointer when moving by 1 row 指针移动一行应该增加多少
    N,  # number of columns in X X 的列数
    eps,  # epsilon to avoid division by zero 用于避免除以 0 的 epsilon
    BLOCK_SIZE: tl.constexpr,
):

# N：每行有多少个有效元素。
# stride：从一行的开头到下一行的开头，需要跨过多少个元素。这里通常取 x.stride(0)


    # Map the program id to the row of X and Y it should compute.
    # 映射程序 id 到对应计算的 X 和 Y 的行
    row = tl.program_id(0)
    Y += row * stride
    X += row * stride
    # Compute mean
    # 计算均值
    mean = 0        #最终存放这一行的均值，是一个标量。
    _mean = tl.zeros([BLOCK_SIZE], dtype=tl.float32)    #存放计算过程中的部分和，是长度为 BLOCK_SIZE 的向量
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        a = tl.load(X + cols, mask=cols < N, other=0.).to(tl.float32)
        _mean += a
    mean = tl.sum(_mean, axis=0) / N
    # Compute variance
    # 计算方差
    _var = tl.zeros([BLOCK_SIZE], dtype=tl.float32)
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        x = tl.load(X + cols, mask=cols < N, other=0.).to(tl.float32)
        x = tl.where(cols < N, x - mean, 0.)
        _var += x * x
    var = tl.sum(_var, axis=0) / N
    rstd = 1 / tl.sqrt(var + eps)
    # Write mean / rstd
    # 写入 mean / rstd
    #为什么要保存？主要是为了反向传播复用
    # LayerNorm 反向传播需要用到前向计算中的统计量，
    tl.store(Mean + row, mean)
    tl.store(Rstd + row, rstd)
    # Normalize and apply linear transformation
    # 归一化并应用线性变换
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N
        w = tl.load(W + cols, mask=mask)
        b = tl.load(B + cols, mask=mask)
        x = tl.load(X + cols, mask=mask, other=0.).to(tl.float32)
        x_hat = (x - mean) * rstd
        y = x_hat * w + b
        # Write output
        tl.store(Y + cols, y, mask=mask)



## 反向传播

LayerNorm 的反向传播需要计算输入、权重和偏置的梯度，比前向传播更复杂。以下先讨论一行包含 $N$ 个元素的情况，再扩展到 $M$ 行输入。

### 符号与形状

设 $L$ 为标量损失，输入矩阵 $X$ 的形状为 $(M, N)$。对其中一行：

| 符号 | 含义 | 逻辑形状 |
|---|---|---|
| $x$、$y$ | 当前行的输入、输出 | $(N,)$ |
| $w$、$b$ | 所有行共享的权重、偏置 | $(N,)$ |
| $\mu$、$\sigma$、$\mathrm{rstd}$ | 当前行的均值、加入 $\epsilon$ 后的标准差及其倒数 | 标量 |
| $\hat{x}$ | 仿射变换前的标准化输入 | $(N,)$ |
| $\nabla_y = \partial L / \partial y$ | 从后续算子传来的输出梯度 | $(N,)$ |

$$
\mu = \frac{1}{N}\sum_{j=0}^{N-1}x_j,
\qquad
\sigma = \sqrt{\frac{1}{N}\sum_{j=0}^{N-1}(x_j-\mu)^2 + \epsilon},
\qquad
\mathrm{rstd} = \frac{1}{\sigma}
$$

$$
\hat{x} = (x-\mu)\,\mathrm{rstd},
\qquad
y = \hat{x}\odot w+b
$$

其中 $\odot$ 表示逐元素相乘，$\cdot$ 表示向量点积。这里将 $\epsilon$ 明确包含在 $\sigma$ 中，与前向代码 `rstd = 1 / tl.sqrt(var + eps)` 保持一致。

来源：用户提供的课程截图；公式与实现对照 [Triton 官方 Layer Normalization 教程](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html#backward-pass)。

### 输入梯度：先做行内归约

先定义仿射变换传回标准化输入的梯度：

$$
g = \nabla_y\odot w
$$

再计算当前行的两个中间标量：

$$
c_1 = \frac{1}{N}\,\hat{x}\cdot g
    = \frac{1}{N}\sum_{j=0}^{N-1}\hat{x}_j g_j,
\qquad
c_2 = \frac{1}{N}\,\nabla_y\cdot w
    = \frac{1}{N}\sum_{j=0}^{N-1}g_j
$$

输入的向量–雅可比积（VJP），即传回输入的梯度，为：

$$
\nabla_x
= \frac{1}{\sigma}
\left(
\nabla_y\odot w
- \underbrace{\left(\frac{1}{N}\hat{x}\cdot(\nabla_y\odot w)\right)}_{c_1}\hat{x}
- \underbrace{\frac{1}{N}\nabla_y\cdot w}_{c_2}
\right)
$$

也就是：

$$
\nabla_x = \mathrm{rstd}\left(g-c_1\hat{x}-c_2\right)
$$

$c_1$、$c_2$ 是**当前行**的中间标量，不同输入行通常有不同的值。标量与向量运算时，会广播到该行的所有有效位置。

对应的 Triton 计算片段如下，供后续反向 kernel 实现时对照；此处使用 Markdown 代码块，不作为可直接执行的完整单元格：

```python
# x、dy、w、cols 的逻辑形状均为 (BLOCK_SIZE_N,)
# mean、rstd 是从前向保存的统计量中加载的标量
mask = cols < N
xhat = tl.where(mask, (x - mean) * rstd, 0.0)
wdy = tl.where(mask, w * dy, 0.0)

c1 = tl.sum(xhat * wdy, axis=0) / N
c2 = tl.sum(wdy, axis=0) / N
dx = (wdy - (xhat * c1 + c2)) * rstd
```

- 两次 `tl.sum(..., axis=0)` 都是在当前 program 内沿特征维度归约，将向量变成标量；一个 program 负责一行。
- 补齐到 `BLOCK_SIZE_N` 的无效位置必须置零，避免污染归约；分母是实际特征数 $N$。
- `mean`、`rstd` 可通过 `tl.load(Mean + row)` 和 `tl.load(Rstd + row)` 加载，因此前向需要保存它们。
- 不需要显式构造形状为 $(N, N)$ 的雅可比矩阵；按上述公式即可计算 VJP。

### 权重与偏置梯度：区分单行贡献和最终结果

对第 $m$ 行，权重和偏置的梯度贡献分别为：

$$
\Delta w^{(m)} = \nabla_{y^{(m)}}\odot\hat{x}^{(m)},
\qquad
\Delta b^{(m)} = \nabla_{y^{(m)}}
$$

截图中的 $\nabla_w=\nabla_y\odot\hat{x}$、$\nabla_b=\nabla_y$ 对应这里的**单行贡献**。由于所有行共享同一组 $w$、$b$，最终梯度必须沿行维度累加：

$$
\nabla_w = \sum_{m=0}^{M-1}\Delta w^{(m)},
\qquad
\nabla_b = \sum_{m=0}^{M-1}\Delta b^{(m)}
$$

对于每个特征位置 $j$：

$$
\frac{\partial L}{\partial w_j}
= \sum_{m=0}^{M-1}
\frac{\partial L}{\partial Y_{m,j}}\hat{X}_{m,j},
\qquad
\frac{\partial L}{\partial b_j}
= \sum_{m=0}^{M-1}\frac{\partial L}{\partial Y_{m,j}}
$$

| 结果 | 形状 | 归约方向 |
|---|---|---|
| $\nabla_X$ | $(M, N)$ | 每行内部计算 $c_1$、$c_2$，再得到该行输入梯度 |
| $\nabla_w$、$\nabla_b$ | 各为 $(N,)$ | 将所有行对同一特征位置的贡献相加 |

因此，输入梯度可以按行独立计算；权重和偏置梯度需要多个 program 协作归约。

\[
\nabla_w^{(m)}=\nabla_y^{(m)}\odot\hat{x}^{(m)}
\]

\[
\nabla_b^{(m)}=\nabla_y^{(m)}
\]

### 两阶段并行归约

令 $G=\texttt{GROUP\_SIZE\_M}$。教程先将多行贡献累加到 $G$ 组部分和缓冲区，再启动另一个 kernel 沿缓冲区维度求和。

![LayerNorm 权重梯度的两阶段并行归约：M=4，GROUP_SIZE_M=2](../images/05-layernorm-parallel-reduction.png)

图源：[Triton 官方教程中的并行归约图](https://triton-lang.org/main/_images/parallel_reduction.png)。图片保存在项目的 `images/` 目录，通过相对路径引用，可离线查看；为简洁起见，图中仅展示权重梯度。



In [ ]:
@triton.jit
def _layer_norm_bwd_dx_fused(DX,  # pointer to the input gradient 输入梯度指针
                             DY,  # pointer to the output gradient 输出梯度指针
                             DW,  # pointer to the partial sum of weights gradient 权重和梯度指针
                             DB,  # pointer to the partial sum of biases gradient 偏差梯度部分和指针
                             X,  # pointer to the input 输入指针
                             W,  # pointer to the weights 权重指针
                             Mean,  # pointer to the mean 均值指针
                             Rstd,  # pointer to the 1/std 1/std 指针
                             Lock,  # pointer to the lock 锁指针
                             stride,  # how much to increase the pointer when moving by 1 row 指针移动一行应该增加多少
                             N,  # number of columns in X X 的列数
                             GROUP_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr):
    '''
    LayerNorm 反向传播的第一阶段 kernel，主要完成两件事：
    1. 计算并保存输入梯度 dx。
    2. 计算权重、偏置梯度的单行贡献，并累加到部分和缓冲区，供第二阶段归约。
    
    '''

    
    # Map the program id to the elements of X, DX, and DY it should compute.
    # 映射程序 id 到对应计算的 X, DX, DY
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE_N)
    mask = cols < N
    X += row * stride   #表示某一行开始的指针
    DY += row * stride
    DX += row * stride
    # Offset locks and weights/biases gradient pointer for parallel reduction
    # 偏移锁和权重/偏差梯度指针以并行归约
    lock_id = row % GROUP_SIZE_M
    Lock += lock_id #找到当前 Lock 的地址
    Count = Lock + GROUP_SIZE_M     #锁数组前两个元素保存 Lock，后两个元素保存 Count，所以只需要从当前 Lock 地址向后移动两个元素，就能找到对应的 Count。
    DW = DW + lock_id * N + cols
    DB = DB + lock_id * N + cols
    # Load data to SRAM
    # 读取数据到 SRAM
    x = tl.load(X + cols, mask=mask, other=0).to(tl.float32)
    dy = tl.load(DY + cols, mask=mask, other=0).to(tl.float32)
    w = tl.load(W + cols, mask=mask).to(tl.float32)
    mean = tl.load(Mean + row)
    rstd = tl.load(Rstd + row)
    # Compute dx
    # 计算 ds
    xhat = (x - mean) * rstd
    wdy = w * dy
    xhat = tl.where(mask, xhat, 0.)
    wdy = tl.where(mask, wdy, 0.)
    c1 = tl.sum(xhat * wdy, axis=0) / N
    c2 = tl.sum(wdy, axis=0) / N
    dx = (wdy - (xhat * c1 + c2)) * rstd
    # Write dx
    # 写入 dx
    tl.store(DX + cols, dx, mask=mask)
    # Accumulate partial sums for dw/db
    # 累加 dw 和 db 的部分和
    partial_dw = (dy * xhat).to(w.dtype)
    partial_db = (dy).to(w.dtype)
    while tl.atomic_cas(Lock, 0, 1) == 1:
        pass
    count = tl.load(Count)
    # First store doesn't accumulate
    # 第一个储存不累加
    if count == 0:
        tl.atomic_xchg(Count, 1)    #以原子方式，将 Count 指向的内存值设为 1，并返回修改前的旧值
    else:
        partial_dw += tl.load(DW, mask=mask)
        partial_db += tl.load(DB, mask=mask)
    tl.store(DW, partial_dw, mask=mask)     #(GROUP_SIZE_M, N)
    tl.store(DB, partial_db, mask=mask)
    # Release the lock
    # 释放锁
    tl.atomic_xchg(Lock, 0)




@triton.jit
def _layer_norm_bwd_dwdb(DW,  # pointer to the partial sum of weights gradient 权重部分和指针
                         DB,  # pointer to the partial sum of biases gradient 偏差梯度部分和指针
                         FINAL_DW,  # pointer to the weights gradient 权重梯度指针
                         FINAL_DB,  # pointer to the biases gradient 偏差梯度指针
                         M,  # GROUP_SIZE_M
                         N,  # number of columns 列数
                         BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr):
    # Map the program id to the elements of DW and DB it should compute.
    # 映射程序 id 到对应计算的 DW 和 DB
    pid = tl.program_id(0)
    cols = pid * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    dw = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    db = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    # Iterate through the rows of DW and DB to sum the partial sums.
    #迭代通过 DW 和 DB 的行，对部分和进行求和。
    for i in range(0, M, BLOCK_SIZE_M):
        rows = i + tl.arange(0, BLOCK_SIZE_M)
        mask = (rows[:, None] < M) & (cols[None, :] < N)
        offs = rows[:, None] * N + cols[None, :]
        dw += tl.load(DW + offs, mask=mask, other=0.)
        db += tl.load(DB + offs, mask=mask, other=0.)
    # Write the final sum to the output.
    # 将最终结果写入输出
    sum_dw = tl.sum(dw, axis=0)
    sum_db = tl.sum(db, axis=0)
    tl.store(FINAL_DW + cols, sum_dw, mask=cols < N)
    tl.store(FINAL_DB + cols, sum_db, mask=cols < N)

## 二、为什么 Triton 要采用两阶段归约？

**归约本身是数学要求，两阶段归约则是一种 GPU 并行实现策略。**

教程采用：

![两阶段归约流程：Program 0、1、2、3 分别计算 dw0、dw1、dw2、dw3；DW[0] 累加 dw0 和 dw2，DW[1] 累加 dw1 和 dw3；最终梯度为 DW[0] 与 DW[1] 之和。](../images/05-layernorm-two-stage-reduction.png)

图中以 4 个 Program、2 组部分和缓冲区为例，$dw_m$ 表示第 $m$ 行对权重梯度的贡献：

$$
\mathrm{DW}[0] = dw_0 + dw_2,
\qquad
\mathrm{DW}[1] = dw_1 + dw_3
$$

$$
\nabla_w = \mathrm{DW}[0] + \mathrm{DW}[1]
$$

这样设计有三个原因：

1. **保持并行度。** 多个 Program 可以同时计算不同行的 `dx`、`partial_dw`、`partial_db`。
2. **减少集中写入竞争。** 使用多个 DW/DB 缓冲区，而不是让所有 Program 都更新同一个最终梯度向量。
3. **避免大量原子逐元素累加。** 每组在锁保护下累加局部梯度，最后由另一个 Kernel 统一求和。

不过，这个实现也需要锁、临时缓冲区和额外一次 Kernel Launch，因此**不保证一定比其他归约方法快**，实际取决于输入规模和硬件。

*来源：用户提供的课程笔记截图；文字已转为 Markdown，流程图按原图重建并保存在项目的 `images/` 目录，可离线查看。*